# 고객문의 텍스트 수집 및 전처리

**환경:** Google Colab / Jupyter Notebook (Python 3)  
**핵심:** pandas · re · scikit-learn · kiwipiepy  
**결과물:** `cleaned_customer_inquiries.csv`

### 학습 목표
1. 고객문의 데이터를 불러와 결측치·중복을 점검한다.
2. 문장, 공백 기반 단어, scikit-learn 토큰을 비교한다.
3. 정규표현식으로 개인정보 마스킹·문자 정제를 수행한다.
4. 정규화·불용어 처리와 한국어 형태소 분석의 차이를 이해한다.
5. scikit-learn에 Kiwi 토크나이저를 연결하고 CSV로 저장한다.

> 교육용 **가상 고객문의 30건**을 사용하며, 외부 파일 업로드는 필요하지 않습니다. 형태소 분석기 설치는 최초 1회 인터넷 연결이 필요합니다. 아래 셀을 위에서부터 순서대로 실행하세요.

## 1. 환경 준비 및 데이터 확인

### 실습 흐름
`패키지 준비 → 예제 데이터 30건 생성 → 데이터 품질 확인 → 분석용 복사본 생성`

**핵심 기능**
- `importlib.util.find_spec()` : 패키지 설치 여부 확인
- `pd.DataFrame()` : 문장 목록을 표 형태로 구성
- `isna()` / `duplicated()` : 결측값과 중복값 확인
- `dropna()` / `drop_duplicates()` : 분석에서 제외할 행 정리

**확인할 점:** 실습 데이터는 생성된 가상 데이터이므로 별도 업로드 없이 진행합니다.

In [2]:
# ── 실습에 필요한 외부 패키지 설치 (처음 한 번만 실행) ──
# 맨 앞의 '!' : 이 줄을 파이썬 코드가 아닌 셸(터미널) 명령으로 실행하라는 노트북 전용 표시
# pip install : 파이썬 패키지를 인터넷에서 내려받아 설치하는 명령
# -q          : 설치 과정에서 출력되는 메시지를 줄여서 표시하는 옵션(quiet)
# kiwipiepy   : 한국어 형태소 분석기 Kiwi를 파이썬에서 사용할 수 있게 해 주는 패키지
# openpyxl    : pandas가 엑셀(.xlsx) 파일을 읽고 쓸 때 내부적으로 사용하는 패키지
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 80.2 MB/s eta 0:00:00


In [3]:
# ── 표준 라이브러리 불러오기 (파이썬에 기본으로 들어 있어 따로 설치하지 않아도 되는 모듈) ──
import importlib.util  # 특정 모듈(패키지)이 설치되어 있는지 확인하는 기능을 제공하는 모듈
import subprocess      # 파이썬 코드 안에서 pip 같은 외부 명령을 실행하는 기능을 제공하는 모듈
import sys             # 현재 실행 중인 파이썬 인터프리터의 경로 등 시스템 정보를 제공하는 모듈


# ── 실습에 사용할 라이브러리 불러오기 ──
import pandas as pd                                          # 표(행·열) 형태의 데이터를 다루는 라이브러리, 'pd'라는 짧은 이름으로 사용
import re                                                    # 정규표현식(문자열 패턴 검색·치환) 기능을 제공하는 표준 모듈
from sklearn.feature_extraction.text import CountVectorizer  # 문장을 토큰으로 나누고 단어 빈도를 세는 scikit-learn 클래스
from kiwipiepy import Kiwi                                   # 한국어 문장을 형태소 단위로 나누고 품사를 붙여 주는 분석기 클래스

# ── 불러오기 결과 확인 출력 ──
# pd.__version__ : 현재 설치된 pandas의 버전 문자열
print("pandas:", pd.__version__)
# 위의 import 문이 모두 오류 없이 실행되었을 때 출력되는 안내 문장
print("실습 환경 준비 완료")

pandas: 2.2.3
실습 환경 준비 완료


In [4]:
# ── 교육용 가상 고객문의 30건 ──
# messages : 고객문의 문장 30개를 순서대로 담은 리스트(list)
# 느낌표 반복(!!!), 이모티콘(ㅠㅠ), 이메일, 전화번호, 이중 공백 등 정제 실습에 쓰일 표현이 섞여 있음
messages = [
 "배송이 아직 안왔어요!!! 확인 부탁드립니다.",
 "주문한 상품 환불하고 싶습니다.",
 "카드 결제가 두 번 되었어요 ㅠㅠ",
 "상품 교환은 어떻게 하나요???",
 "배송 문의는 test@example.com으로 연락주세요.",
 "배송이 너무 늦어요. 언제 도착하나요?",
 "결제 오류가 계속 발생합니다.",
 "환불 금액이 아직 입금되지 않았습니다.",
 "상품 사이즈를 교환하고 싶어요.",
 "택배배송 조회가 안 됩니다.",
 "배송 주소를 변경할 수 있나요?",
 "결제 취소했는데 카드 승인 내역이 남아 있어요.",
 "제품이 파손되어 도착했습니다. 교환 부탁드려요.",
 "반품 신청은 어디에서 하나요?",
 "영수증을 이메일로 보내 주세요.",
 "배송  지연으로  너무  불편합니다!!!",
 "쿠폰 적용이 안 되어 결제 금액이 달라요.",
 "환불 처리 기간이 궁금합니다.",
 "구매한 물건 색상이 다릅니다.",
 "배송 완료라는데 상품을 못 받았어요.",
 "AS문의는 어디에 접수하나요?",
 "주문번호 12345 배송 확인 부탁드립니다.",
 "010-1234-5678로 연락 부탁드립니다.",
 "상담원이 안내한 교환 방법을 알고 싶어요.",
 "결제 수단을 바꾸고 싶습니다.",
 "상품이 누락되었습니다. 재배송 요청합니다.",
 "안녕하세요. 환불 신청을 취소하고 싶어요.",
 "정비 매뉴얼 파일을 이메일로 받을 수 있을까요?",
 "배송이 늦어서 환불하고 싶습니다.",
 "제품 배송 현황을 어디서 확인하나요?",
]

# ── 문의번호와 원본텍스트로 DataFrame 생성 ──
# pd.DataFrame({...}) : 딕셔너리의 키를 열 이름, 값을 열 데이터로 하는 표를 만듦
# range(1, len(messages)+1) : 1부터 문장 개수(30)까지의 정수 → 문의번호 열
# len(messages) : 리스트에 들어 있는 문장의 개수
df = pd.DataFrame({"문의번호": range(1, len(messages)+1), "원본텍스트": messages})

# ── 데이터 크기와 앞부분 8행 출력 ──
# df.shape : (행 개수, 열 개수) 형태의 튜플
print("데이터 크기:", df.shape)
# df.head(8) : 표의 맨 앞 8행만 잘라낸 결과
# display() : 노트북에서 DataFrame을 표 모양으로 보기 좋게 출력하는 함수
display(df.head(8))

데이터 크기: (30, 2)


,문의번호,원본텍스트
0,1,배송이 아직 안왔어요!!! 확인 부탁드립니다.
1,2,주문한 상품 환불하고 싶습니다.
2,3,카드 결제가 두 번 되었어요 ㅠㅠ
3,4,상품 교환은 어떻게 하나요???
4,5,배송 문의는 test@example.com으로 연락주세요.
5,6,배송이 너무 늦어요. 언제 도착하나요?
6,7,결제 오류가 계속 발생합니다.
7,8,환불 금액이 아직 입금되지 않았습니다.


In [6]:
# ── 결측값과 중복 문의 개수 출력 ──
# df["원본텍스트"]       : 표에서 '원본텍스트' 열만 꺼낸 시리즈(Series)
# .isna()               : 각 값이 비어 있으면(NaN) True, 아니면 False로 바꾼 결과
# .sum()                : True를 1, False를 0으로 계산해 더한 값 → 비어 있는 값의 개수
print("결측값 수:", df["원본텍스트"].isna().sum())
# .duplicated()         : 앞에서 이미 나온 값과 같으면 True로 표시한 결과
# .sum()                : 중복으로 표시된 값의 개수
print("중복 문의 수:", df["원본텍스트"].duplicated().sum())

# ── 결측·중복 행을 제외한 작업용 복사본 생성 (원본 df는 유지) ──
# .dropna(subset=["원본텍스트"])          : '원본텍스트' 열이 비어 있는 행을 제거
# .drop_duplicates(subset=["원본텍스트"]) : '원본텍스트' 열 값이 같은 행 중 첫 번째만 남기고 제거
# .copy()                                : 원본 df와 연결되지 않은 별도의 표로 복사
work = df.dropna(subset=["원본텍스트"]).drop_duplicates(subset=["원본텍스트"]).copy()

# ── 텍스트 컬럼을 문자열 타입으로 변환 ──
# .astype(str) : 열의 모든 값을 문자열(str) 자료형으로 바꿈
work["원본텍스트"] = work["원본텍스트"].astype(str)

# ── 정리 후 행 수 출력 ──
# len(work) : 작업용 표의 행 개수
print("정리 후 데이터 수:", len(work))

결측값 수: 0
중복 문의 수: 0
정리 후 데이터 수: 30


## 2. 문장·단어 단위 처리

- **문장 분리**: 문장 종결부호 기준의 단순 규칙(완전한 한국어 문장분리기는 아님)
- **공백 단위**: 조사·어미가 어절에 붙어 있음
- **CountVectorizer**: 기본 정규표현식은 글자 2개 이상인 토큰을 추출하므로 한 글자 토큰이 제외될 수 있음

### 실습 흐름
`예제 문장 → 문장 분리 → 공백 분리 → scikit-learn 기본 토큰 확인 → 결과 비교`

**핵심 기능**
- `re.split()` : 지정한 문장 경계 규칙에 따라 문자열 분리
- `str.split()` : 공백을 기준으로 어절 분리
- `CountVectorizer().build_analyzer()` : scikit-learn의 전처리·토큰화 결과 확인

**확인할 점:** `CountVectorizer`는 한국어 조사와 어미를 자동으로 형태소 분석하지 않습니다. 문장 분리 정규식도 교육용 단순 규칙입니다.

In [10]:
# ── 비교에 사용할 예제 문장 ──
# 마침표(.)와 물음표(?)가 포함된 두 문장으로 된 문자열
sample = "배송이 늦었습니다. 언제 받을 수 있나요?"

# ── 공백 기준 어절 분리 ──
# str.split() : 괄호 안을 비우면 공백(띄어쓰기·탭·줄바꿈)을 기준으로 문자열을 나눠 리스트로 반환
# 문장부호(., ?)는 앞 단어에 붙은 채로 남음
words = sample.split()

# ── CountVectorizer 기본 설정의 토큰화 함수 ──
# CountVectorizer()   : 아무 옵션 없이 만든 기본 설정의 벡터라이저 객체
# .build_analyzer()   : 문자열 하나를 넣으면 전처리(소문자화)와 토큰화를 거친 토큰 리스트를 돌려주는 함수를 반환
# 기본 토큰 규칙은 글자 2개 이상인 단어만 추출하므로 한 글자 토큰('수')과 문장부호는 빠짐
analyzer = CountVectorizer().build_analyzer()

# ── 세 방식의 결과 출력 ──
print("문장:", sample)                          # 원본 문장
print("공백 기반:", words)                      # 공백 분리 결과 리스트
print("scikit-learn 기본:", analyzer(sample))  # analyzer 함수에 문장을 넣어 얻은 토큰 리스트

문장: 배송이 늦었습니다. 언제 받을 수 있나요?
공백 기반: ['배송이', '늦었습니다.', '언제', '받을', '수', '있나요?']
scikit-learn 기본: ['배송이', '늦었습니다', '언제', '받을', '있나요']


## 3. 정규표현식으로 텍스트 정제


### 실습 흐름
`원본 텍스트 → 이메일·전화번호 치환 → 특수문자 정리 → 연속 공백 정리 → 원본과 비교`

**핵심 기능**
- `re.sub(패턴, 대체문자열, 텍스트)` : 패턴에 맞는 부분 치환
- `apply(함수)` : DataFrame의 각 문장에 사용자 정의 함수 적용
- `display()` / `head()` : 전후 결과를 표로 확인

**확인할 점:** 개인정보에 해당하는 패턴부터 치환합니다. 이 예제의 정규표현식은 대표적인 형식만 처리하며 실제 서비스의 모든 형태를 포괄하지 않습니다.

In [16]:
# ── 개인정보 마스킹 → 특수문자 제거 → 공백 정리를 수행하는 함수 ──
# def 함수이름(매개변수): 사용자 정의 함수를 만드는 문법, text에 문장 하나가 들어옴
def clean_text(text):
    # 입력값을 문자열로 변환 (숫자 등 다른 자료형이 들어와도 아래 처리가 가능하도록 함)
    text = str(text)

    # 이메일 주소를 임시 토큰으로 치환
    # re.sub(패턴, 바꿀문자열, 대상) : 대상 문자열에서 패턴과 일치하는 부분을 모두 바꿈
    # [\w.+-]+      : 영문·숫자·밑줄(\w)과 . + - 문자가 1개 이상 이어진 부분 → 아이디 부분
    # @             : 골뱅이 문자
    # [\w.-]+       : 영문·숫자·밑줄과 . - 가 1개 이상 → 도메인 이름 부분
    # \.[A-Za-z]{2,} : 마침표 뒤에 영문자 2개 이상 → .com, .kr 같은 최상위 도메인
    # 앞뒤에 공백을 넣은 " EMAILTOKEN "으로 바꿔 주변 글자와 붙지 않게 함
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " EMAILTOKEN ", text)

    # 휴대폰 번호(010 등, 하이픈·공백 선택)를 임시 토큰으로 치환
    # (?<!\d)      : 바로 앞 글자가 숫자가 아닐 때만 일치 (더 긴 숫자열의 일부는 제외)
    # 01[016789]   : 010, 011, 016, 017, 018, 019 중 하나
    # [- ]?        : 하이픈 또는 공백이 0개 또는 1개
    # \d{3,4}      : 숫자 3~4자리 (가운데 번호)
    # \d{4}        : 숫자 4자리 (끝 번호)
    # (?!\d)       : 바로 뒤 글자가 숫자가 아닐 때만 일치
    text = re.sub(r"(?<!\d)01[016789][- ]?\d{3,4}[- ]?\d{4}(?!\d)", " PHONETOKEN ", text)

    # 한글·영문·숫자·공백 외 문자를 공백으로 치환
    # [^...]   : 대괄호 안에 나열한 문자가 '아닌' 문자 1개
    # 가-힣     : 완성형 한글 글자 범위
    # A-Za-z   : 영문 대문자·소문자
    # 0-9      : 숫자
    # \s       : 공백 문자(띄어쓰기·탭·줄바꿈)
    # 느낌표, 물음표, 마침표, 'ㅠㅠ' 같은 자음·모음 단독 글자가 이 단계에서 공백으로 바뀜
    text = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", text)

    # 연속 공백을 하나로 줄이고 앞뒤 공백 제거
    # \s+      : 공백 문자가 1개 이상 연속된 부분
    # .strip() : 문자열 맨 앞과 맨 뒤의 공백을 제거
    text = re.sub(r"\s+", " ", text).strip()

    # 임시 토큰을 [EMAIL], [PHONE] 표기로 변환해 반환
    # 대괄호가 위 특수문자 제거 단계에서 지워지지 않도록 마지막에 바꿈
    # .replace(찾을문자열, 바꿀문자열) : 문자열 안의 일치하는 부분을 모두 바꿈
    return text.replace("EMAILTOKEN", "[EMAIL]").replace("PHONETOKEN", "[PHONE]")

# ── 모든 행에 정제 함수 적용 ──
# 결과를 '정제텍스트'라는 새 열로 work 표에 추가
work["정제텍스트"] = work["원본텍스트"].apply(clean_text)
# .apply(clean_text) 각 원소를 clean_text에 하나씩 넣어 호출하고, 반환값들을 모아 새 값으로 만듦

# ── 원본과 정제 결과 비교 출력 ──
# work[["원본텍스트", "정제텍스트"]] : 대괄호 안에 열 이름 리스트를 넣어 두 열만 선택한 표
display(work[["원본텍스트", "정제텍스트"]])


,원본텍스트,정제텍스트
0,배송이 아직 안왔어요!!! 확인 부탁드립니다.,배송이 아직 안왔어요 확인 부탁드립니다
1,주문한 상품 환불하고 싶습니다.,주문한 상품 환불하고 싶습니다
2,카드 결제가 두 번 되었어요 ㅠㅠ,카드 결제가 두 번 되었어요
3,상품 교환은 어떻게 하나요???,상품 교환은 어떻게 하나요
4,배송 문의는 test@example.com으로 연락주세요.,배송 문의는 [EMAIL] 으로 연락주세요
5,배송이 너무 늦어요. 언제 도착하나요?,배송이 너무 늦어요 언제 도착하나요
6,결제 오류가 계속 발생합니다.,결제 오류가 계속 발생합니다
7,환불 금액이 아직 입금되지 않았습니다.,환불 금액이 아직 입금되지 않았습니다
8,상품 사이즈를 교환하고 싶어요.,상품 사이즈를 교환하고 싶어요
9,택배배송 조회가 안 됩니다.,택배배송 조회가 안 됩니다


## 4. 표현 정규화 및 불용어 처리

- 표기 변형을 통일하면 동일한 키워드를 비교하기 쉬워집니다.
- 불용어는 **업무 목적에 따라** 결정합니다. 특히 `안`, `못`, `없다` 같은 부정 표현은 함부로 제거하지 않습니다.

### 실습 흐름
`정제 문장 → 표기 통일 → 불용어 목록 정의 → 처리 전후 토큰 비교`

**핵심 기능**
- `str.replace()` : 특정 표현을 통일된 표현으로 교체
- `CountVectorizer(stop_words=...)` : 지정한 토큰을 분석 과정에서 제외
- `build_analyzer()` : 불용어가 적용된 토큰 목록 확인

**확인할 점:** 불용어 제거는 원본 문장을 직접 삭제하는 것이 아니라 **토큰 분석 단계**에서 적용됩니다. 부정 표현은 중요한 의미를 담을 수 있습니다.

In [18]:
# ── 지정한 표기 변형을 통일하는 함수 ──
# 같은 뜻인데 띄어쓰기가 다른 표현을 하나의 형태로 맞춤
def normalize_text(text):
    # 붙어 있는 복합어를 띄어 쓴 형태로 통일
    # "택배배송" → "택배 배송"
    text = text.replace("택배배송", "택배 배송")
    # "AS문의" → "AS 문의"
    text = text.replace("AS문의", "AS 문의")
    # 연속 공백을 하나로 줄이고 앞뒤 공백 제거
    text = re.sub(r"\s+", " ", text).strip()
    # 정규화된 문자열 반환
    return text

# ── 모든 행에 정규화 함수 적용 ──
# '정제텍스트' 열에 함수를 적용한 결과를 '정규화텍스트' 열로 추가
work["정규화텍스트"] = work["정제텍스트"].apply(normalize_text)
# .apply(normalize_text) 각 원소를 normalize_text에 하나씩 넣어 호출하고, 반환값들을 모아 새 값으로 만듦

# ── 어절 단위 불용어 목록 ──
# 분석에서 제외할 단어 목록 (인사말·요청 표현)
# CountVectorizer는 토큰과 '완전히 같은' 문자열만 불용어로 제거함
stop_words = ["안녕하세요", "부탁드립니다", "부탁드려요"]

# ── 불용어를 적용한 CountVectorizer 생성 ──
# stop_words=stop_words : 위 목록에 있는 토큰을 토큰화 결과에서 제외하도록 설정
cv = CountVectorizer(stop_words=stop_words)

# ── 불용어 적용 전후 토큰 비교 ──
# 비교용 예제 문장
stop_test = "안녕하세요 배송 확인 부탁드립니다"
# 불용어 설정 없는 기본 analyzer로 토큰화한 결과 출력
print("불용어 처리 전:", CountVectorizer().build_analyzer()(stop_test))
# 불용어 설정을 적용한 cv의 analyzer로 토큰화한 결과 출력
print("불용어 처리 후:", cv.build_analyzer()(stop_test))

# ── 정제 텍스트와 정규화 텍스트 비교 출력 ──
# .head(10) : 앞 10행만 표시
display(work[["정제텍스트", "정규화텍스트"]].head(10))

불용어 처리 전: ['안녕하세요', '배송', '확인', '부탁드립니다']
불용어 처리 후: ['배송', '확인']


,정제텍스트,정규화텍스트
0,배송이 아직 안왔어요 확인 부탁드립니다,배송이 아직 안왔어요 확인 부탁드립니다
1,주문한 상품 환불하고 싶습니다,주문한 상품 환불하고 싶습니다
2,카드 결제가 두 번 되었어요,카드 결제가 두 번 되었어요
3,상품 교환은 어떻게 하나요,상품 교환은 어떻게 하나요
4,배송 문의는 [EMAIL] 으로 연락주세요,배송 문의는 [EMAIL] 으로 연락주세요
5,배송이 너무 늦어요 언제 도착하나요,배송이 너무 늦어요 언제 도착하나요
6,결제 오류가 계속 발생합니다,결제 오류가 계속 발생합니다
7,환불 금액이 아직 입금되지 않았습니다,환불 금액이 아직 입금되지 않았습니다
8,상품 사이즈를 교환하고 싶어요,상품 사이즈를 교환하고 싶어요
9,택배배송 조회가 안 됩니다,택배 배송 조회가 안 됩니다


## 5. 한국어 형태소 분석과 scikit-learn 연동

Kiwi는 **형태소 분석**을 담당하고 scikit-learn의 `CountVectorizer`는 **분석 결과를 받아 토큰화 인터페이스와 문서-단어 행렬**을 제공합니다.
토큰 목록 비교에 초점을 둡니다.

### 실습 흐름
`예제 문장 → Kiwi 형태소·품사 확인 → 필요한 품사 선택 → 사용자 정의 토크나이저 → scikit-learn 연결`

**핵심 기능**
- `Kiwi()` / `kiwi.tokenize()` : 한국어 형태소와 품사 분석
- `token.form` / `token.tag` : 형태소 표면형과 품사 태그
- `CountVectorizer(tokenizer=...)` : 외부 형태소 분석 함수를 토큰화 과정에 사용

**토큰 선택 규칙 (교육용 예시)**
- 유지 품사: `NNG`(일반 명사), `NNP`(고유 명사), `VV`(동사), `VA`(형용사), `SL`(알파벳 — `AS`, 마스킹 토큰 `EMAIL`·`PHONE` 포함)
- 부정 표현 보존: `안`·`못`(MAG), `않`(VX), `없`(VA)
- 불용어는 Kiwi 분석 결과 형태 기준 `(형태, 품사)` 조합으로 지정: `안녕/NNG`, `부탁/NNG`, `드리/VV`
- Kiwi는 동사·형용사 태그에 불규칙 활용 표시(`-R`, `-I`)를 붙이므로(예: `받/VV-R`) 태그 비교 전에 기본 태그로 변환
- 용언은 어간 뒤에 `다`를 붙여 기본형으로 표기(예: `받` → `받다`)

**확인할 점:** 부정 표현 보존은 문의 내용 해석에 유리하지만, 키워드 빈도 분석에서는 `안`·`못`이 상위 빈도로 나타날 수 있습니다. 품사·불용어 정책은 분석 목적에 따라 조정합니다.

In [19]:
# ── Kiwi 형태소 분석기 객체 생성 ──
# Kiwi() : 기본 모델을 불러와 형태소 분석기를 준비함 (모델 로딩에 몇 초가 걸릴 수 있음)
kiwi = Kiwi()

# ── 비교에 사용할 예제 문장 ──
example = "배송이 늦어서 환불하고 싶습니다."

# ── 공백 분리 결과 ──
# 띄어쓰기 단위(어절)로 나눈 결과, 조사·어미가 단어에 붙어 있음
print("공백 분리:", example.split())

# ── CountVectorizer 기본 토큰화 결과 ──
# 2글자 이상 단어를 추출하고 문장부호를 제거한 결과, 조사·어미는 그대로 붙어 있음
print("CountVectorizer 기본:", CountVectorizer().build_analyzer()(example))

# ── Kiwi 형태소(form)와 품사(tag) 결과 ──
# kiwi.tokenize(문장) : 문장을 형태소 단위 토큰 목록으로 분석해 반환
# t.form : 토큰의 글자 형태 (예: '배송')
# t.tag  : 토큰의 품사 태그 (예: 'NNG' 일반 명사, 'JKS' 주격 조사, 'VA' 형용사, 'EC' 연결 어미)
# [(t.form, t.tag) for t in ...] : 각 토큰을 (형태, 품사) 튜플로 바꿔 리스트로 만드는 리스트 컴프리헨션
print("Kiwi 형태소(형태, 품사):", [(t.form, t.tag) for t in kiwi.tokenize(example)])

공백 분리: ['배송이', '늦어서', '환불하고', '싶습니다.']
CountVectorizer 기본: ['배송이', '늦어서', '환불하고', '싶습니다']
Kiwi 형태소(형태, 품사): [('배송', 'NNG'), ('이', 'JKS'), ('늦', 'VA'), ('어서', 'EC'), ('환불', 'NNG'), ('하', 'XSV'), ('고', 'EC'), ('싶', 'VX'), ('습니다', 'EF'), ('.', 'SF')]


**[맞춤 토크나이저 + scikit-learn 연결]**


한국어 문장을 "의미 있는 형태소만 남긴 토큰 목록"으로 바꾸고, 그것을 scikit-learn의 CountVectorizer에 연결해 단어 빈도 행렬(Bag of Words)을 만들 준비

In [28]:
#[맞춤 토크나이저 + scikit-learn 연결
# ── 유지할 기본 품사 태그 (일반명사, 고유명사, 동사, 형용사, 알파벳) ──
# 중괄호 {} 로 만든 집합(set) : 'in' 으로 포함 여부를 빠르게 확인할 수 있는 자료형
# NNG 일반 명사 / NNP 고유 명사 / VV 동사 / VA 형용사 / SL 알파벳(영문)
keep_tags = {"NNG", "NNP", "VV", "VA", "SL"}

# ── 품사와 관계없이 유지할 부정 표현 (형태, 품사) 조합 ──
# ("안","MAG") 부사 '안' / ("못","MAG") 부사 '못' / ("않","VX") 보조 용언 '않' / ("없","VA") 형용사 '없'
negation_set = {("안", "MAG"), ("못", "MAG"), ("않", "VX"), ("없", "VA")}

# ── 제외할 불용어 (형태, 품사) 조합 - Kiwi 분석 결과 형태 기준 ──
# '안녕하세요'의 '안녕', '부탁드립니다'의 '부탁'과 '드리'를 제외 대상으로 지정
stop_pairs = {("안녕", "NNG"), ("부탁", "NNG"), ("드리", "VV")}

# ── Kiwi 분석 결과에서 필요한 형태소만 골라 반환하는 함수 ──
# text에 문장 하나가 들어오면 선택된 토큰들의 리스트를 반환
def kiwi_tokenizer(text):
    # 선택된 토큰을 차례로 담을 빈 리스트
    tokens = []
    # kiwi.tokenize(text) 결과의 토큰을 하나씩 꺼내 token 변수에 담아 반복
    for token in kiwi.tokenize(text):
        # Kiwi가 반환한 품사 태그를 그대로 저장
        base_tag = token.tag         #'NNG'
        # 형태와 품사를 묶은 비교용 키 (튜플)
        pair = (token.form, base_tag)  #('색상', 'NNG')
        # 불용어 조합이면 건너뜀
        # continue : 아래 코드를 실행하지 않고 다음 토큰으로 넘어감
        if pair in stop_pairs:
            continue
        # 부정 표현이거나 유지 대상 품사이면 토큰 목록에 추가
        # or : 두 조건 중 하나라도 참이면 참
        if pair in negation_set or base_tag in keep_tags:
            # 용언(VV·VA·VX)은 어간 뒤에 "다"를 붙여 기본형으로 표기
            # 예: '늦' → '늦다', '않' → '않다'
            if base_tag in {"VV", "VA", "VX"}:
                tokens.append(token.form + "다")
            # 그 외(명사, 알파벳, 부사 '안'·'못')는 형태 그대로 추가
            else:
                tokens.append(token.form)
    # 선택된 토큰 리스트 반환
    return tokens

# ── 사용자 정의 토크나이저를 연결한 CountVectorizer 생성 ──
vectorizer = CountVectorizer(
    tokenizer=kiwi_tokenizer,  # 토큰화 단계에서 위에서 만든 kiwi_tokenizer 함수를 사용
    token_pattern=None,        # 기본 토큰 정규식을 사용하지 않음 (tokenizer를 지정했을 때 경고가 나오지 않도록 함)
    lowercase=False            # 영문을 소문자로 바꾸지 않고 대소문자를 그대로 유지 (EMAIL, PHONE, AS 유지)
)

# ── 예제 문장의 토큰 결과 출력 ──
# vectorizer.build_analyzer() : 전처리 + kiwi_tokenizer 토큰화를 수행하는 함수를 반환
print("Kiwi 맞춤형 토큰:", vectorizer.build_analyzer()(example))

# ── 전체 데이터에 토크나이저 적용 ──
# '정규화텍스트' 열의 각 문장을 kiwi_tokenizer로 토큰화한 리스트를 '분석토큰' 열로 추가
work["분석토큰"] = work["정규화텍스트"].apply(kiwi_tokenizer)
# .apply(kiwi_tokenizer) 각 원소를 kiwi_tokenizer에 하나씩 넣어 호출하고, 반환값들을 모아 새 값으로 만듦

# ── 정규화 텍스트와 분석 토큰 비교 출력 ──
display(work[["정규화텍스트", "분석토큰"]].head(10))

Kiwi 맞춤형 토큰: ['배송', '늦다', '환불']


,정규화텍스트,분석토큰
0,배송이 아직 안왔어요 확인 부탁드립니다,"[배송, 안, 오다, 확인]"
1,주문한 상품 환불하고 싶습니다,"[주문, 상품, 환불]"
2,카드 결제가 두 번 되었어요,"[카드, 결제, 되다]"
3,상품 교환은 어떻게 하나요,"[상품, 교환, 하다]"
4,배송 문의는 [EMAIL] 으로 연락주세요,"[배송, 문의, EMAIL, 연락, 주다]"
5,배송이 너무 늦어요 언제 도착하나요,"[배송, 늦다, 도착]"
6,결제 오류가 계속 발생합니다,"[결제, 오류, 발생]"
7,환불 금액이 아직 입금되지 않았습니다,"[환불, 금액, 입금, 않다]"
8,상품 사이즈를 교환하고 싶어요,"[상품, 사이즈, 교환]"
9,택배 배송 조회가 안 됩니다,"[택배, 배송, 조회, 안, 되다]"


## 6. 결과 점검과 파일 저장 (5분)

결과 파일은 다음 단원(BoW, N-gram, TF-IDF)에서 재사용할 수 있습니다. 토큰 목록은 CSV 저장에 적합하도록 공백으로 연결하고, 원본도 함께 보존합니다.

### 실습 흐름
`토큰 목록 → 공백 구분 문자열 → 컬럼 선택 → 데이터 검증 → CSV 저장`

**핵심 기능**
- `" ".join(tokens)` : 리스트의 토큰을 하나의 문자열로 합침
- `assert` : 결과가 필수 조건을 만족하는지 확인
- `to_csv(index=False, encoding="utf-8-sig")` : 한국어 CSV 저장

**확인할 점:** 원본 텍스트와 정제 텍스트를 함께 남겨야 데이터 가공 과정을 추적할 수 있습니다.

In [30]:
# ── 토큰 리스트를 공백으로 연결한 문자열 컬럼 생성 ──
# " ".join : 리스트 안의 문자열들을 공백 한 칸을 사이에 두고 하나의 문자열로 합치는 함수
# 예: ['배송', '늦다', '환불'] → '배송 늦다 환불'
# CSV 파일에는 리스트를 그대로 저장하기보다 문자열로 바꿔 저장하는 것이 다루기 쉬움
work["토큰문자열"] = work["분석토큰"].apply(" ".join)

# ── 저장할 컬럼만 선택해 복사 ──
# 원본 → 정제 → 정규화 → 토큰 순서로 가공 단계별 열을 함께 남김
# .copy() : work와 분리된 별도의 표로 복사
output = work[["문의번호", "원본텍스트", "정제텍스트", "정규화텍스트", "토큰문자열"]].copy()

# ── 저장 파일 경로 ──
# 폴더 없이 파일 이름만 쓰면 현재 작업 폴더(코랩은 /content)에 저장됨
path = "cleaned_customer_inquiries.csv"

# ── 행 번호 없이 utf-8-sig 인코딩으로 CSV 저장 ──
# index=False          : DataFrame의 행 번호(0, 1, 2 …)를 파일에 쓰지 않음
# encoding="utf-8-sig" : 엑셀에서 열어도 한글이 깨지지 않도록 BOM이 포함된 UTF-8로 저장
output.to_csv(path, index=False, encoding="utf-8-sig")

# ── 저장 결과 출력 ──
# f"...{변수}..." : 중괄호 안의 변수 값을 문자열에 넣는 f-문자열
print(f"완료: {path} / {len(output)}건")
# 저장한 표 전체를 표 형태로 출력
display(output)

완료: cleaned_customer_inquiries.csv / 30건


,문의번호,원본텍스트,정제텍스트,정규화텍스트,토큰문자열
0,1,배송이 아직 안왔어요!!! 확인 부탁드립니다.,배송이 아직 안왔어요 확인 부탁드립니다,배송이 아직 안왔어요 확인 부탁드립니다,배송 안 오다 확인
1,2,주문한 상품 환불하고 싶습니다.,주문한 상품 환불하고 싶습니다,주문한 상품 환불하고 싶습니다,주문 상품 환불
2,3,카드 결제가 두 번 되었어요 ㅠㅠ,카드 결제가 두 번 되었어요,카드 결제가 두 번 되었어요,카드 결제 되다
3,4,상품 교환은 어떻게 하나요???,상품 교환은 어떻게 하나요,상품 교환은 어떻게 하나요,상품 교환 하다
4,5,배송 문의는 test@example.com으로 연락주세요.,배송 문의는 [EMAIL] 으로 연락주세요,배송 문의는 [EMAIL] 으로 연락주세요,배송 문의 EMAIL 연락 주다
5,6,배송이 너무 늦어요. 언제 도착하나요?,배송이 너무 늦어요 언제 도착하나요,배송이 너무 늦어요 언제 도착하나요,배송 늦다 도착
6,7,결제 오류가 계속 발생합니다.,결제 오류가 계속 발생합니다,결제 오류가 계속 발생합니다,결제 오류 발생
7,8,환불 금액이 아직 입금되지 않았습니다.,환불 금액이 아직 입금되지 않았습니다,환불 금액이 아직 입금되지 않았습니다,환불 금액 입금 않다
8,9,상품 사이즈를 교환하고 싶어요.,상품 사이즈를 교환하고 싶어요,상품 사이즈를 교환하고 싶어요,상품 사이즈 교환
9,10,택배배송 조회가 안 됩니다.,택배배송 조회가 안 됩니다,택배 배송 조회가 안 됩니다,택배 배송 조회 안 되다



**참고 공식 문서**  
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)  
- [scikit-learn Text Analytics Tutorial](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)  
- [Kiwi (kiwipiepy)](https://github.com/bab2min/kiwipiepy)